# 🎙 Транскрибация видео с разделением по спикерам (Colab + Яндекс.Диск)

Кладёшь видео в папку на Яндекс.Диске → запускаешь ячейки сверху вниз → рядом с каждым
видео появляется готовый транскрипт (.docx): «Спикер 1:», «Спикер 2:» и абзацы
с таймкодами — тот же формат, что отдаёт SpeechToText.

Под капотом: WhisperX **large-v3** (транскрибация + чистка галлюцинаций + повторный проход
по проблемным зонам + пословное выравнивание wav2vec2) и
**pyannote speaker-diarization-community-1** (диаризация).

---

## Разовая настройка (5 минут, один раз)

1. **GPU:** меню «Среда выполнения → Сменить среду выполнения» → выбери **T4 GPU**.
2. **Секреты** (значок ключа 🔑 на левой панели, включи каждому «Доступ из блокнота»):
   - `HF_TOKEN` — токен Hugging Face ([создать](https://huggingface.co/settings/tokens), тип Read) — нужен для диаризации;
   - `YANDEX_TOKEN` — OAuth-токен Яндекс.Диска. Получить: зарегистрируй приложение на
     [oauth.yandex.ru](https://oauth.yandex.ru/client/new) с правами «Яндекс.Диск REST API»
     (чтение и запись), затем возьми токен по ссылке вида
     `https://oauth.yandex.ru/authorize?response_type=token&client_id=<ID твоего приложения>`.
3. **Условия моделей диаризации:** зайди под своим HF-аккаунтом и нажми «Agree» на страницах:
   - [pyannote/speaker-diarization-community-1](https://huggingface.co/pyannote/speaker-diarization-community-1) (основная)
   - [pyannote/speaker-diarization-3.1](https://huggingface.co/pyannote/speaker-diarization-3.1) и
     [pyannote/segmentation-3.0](https://huggingface.co/pyannote/segmentation-3.0) (фолбэк)
4. **Папка на Яндекс.Диске:** создай папку (по умолчанию `/Transcribe`) и закинь туда видео.

Дальше — просто «Среда выполнения → Выполнить всё». Скорость на T4: примерно 5–8 минут на час видео
без диаризации и 10–15 минут с ней.

## ⚙️ Настройки запуска
**Меняй только эту ячейку.**

In [ ]:
# ══════════════════════════════════════════════════════════
#  НАСТРОЙКИ ЗАПУСКА
# ══════════════════════════════════════════════════════════

# Папка с видео на Яндекс.Диске (обрабатываются все вложенные подпапки).
# Форматы: mp4, mov, avi, mkv, webm, m4v, zip.
VIDEOS_FOLDER = "/Transcribe"

# Куда класть результат, если видео найдено не через VIDEOS_FOLDER (обычно не нужно —
# по умолчанию транскрипт кладётся РЯДОМ с самим видео на Диске).
OUTPUT_FOLDER = "/Transcribe/_results"

# Разделение по спикерам (True/False). Без него — в ~2 раза быстрее, но реплики не размечены.
DIARIZE = True

# Сколько спикеров ожидается в записи (диапазон). Если не знаешь — оставь как есть.
MIN_SPEAKERS = 1
MAX_SPEAKERS = 6

# Язык речи ("ru", "en", ...)
LANGUAGE = "ru"

# Сколько видео обработать за один запуск (0 = все найденные)
MAX_VIDEOS = 0

# Пропускать видео короче N секунд
MIN_DURATION_SEC = 30

# Подсказка Whisper — ТОЛЬКО ключевые слова темы через запятую, НЕ предложения
# (полные предложения Whisper повторяет при тишине → галлюцинации). "" = без подсказки.
INITIAL_PROMPT = ""

# Модель Whisper: "large-v3" (лучшее качество) | "medium" | "small" (быстрее, хуже)
WHISPER_MODEL = "large-v3"

print("✅ Настройки заданы. Запускай следующие ячейки.")

## 📦 Установка зависимостей и код проекта (~3–4 минуты)

In [ ]:
import subprocess, shutil, os, sys

# Удаляем старую копию репо если есть
REPO_DIR = "/content/colab-drive-transcriber"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("🗑️ Старая копия удалена")

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

# ── Шаг 1: whisperx без torch-зависимостей (используем Colab torch) ──
print("📦 Шаг 1/5: WhisperX (без torch)...")
pip("whisperx", "--no-deps")
pip("faster-whisper", "--no-deps")
pip("ctranslate2", "av", "tokenizers>=0.13")

# ── Шаг 2: pyannote (диаризация) ──
print("📦 Шаг 2/5: Pyannote...")
pip("pyannote.audio>=4.0")
pip("pyannote.core", "pyannote.database", "pyannote.metrics", "pyannote.pipeline")
pip("speechbrain", "asteroid-filterbanks", "torch-audiomentations", "einops", "lightning")

# ── Шаг 3: transformers (совместимая версия) ──
print("📦 Шаг 3/5: Transformers...")
pip("transformers>=4.40,<4.52", "huggingface-hub>=0.20")

# ── Шаг 4: утилиты ──
print("📦 Шаг 4/5: Утилиты...")
pip("noisereduce", "soundfile", "nltk", "pyyaml", "optuna", "hyperpyyaml", "docopt", "rich", "python-docx")

# ── Шаг 5: клиент Яндекс.Диска ──
print("📦 Шаг 5/5: Яндекс.Диск...")
pip("yadisk")

print("✅ Все пакеты установлены")

# ── Клонируем репозиторий ──
print("📥 Клонирование репозитория...")
result = subprocess.run(
    ["git", "clone", "--depth", "1",
     "https://github.com/Hipposum/Hipposum-colab-drive-transcriber.git", REPO_DIR],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("❌ Ошибка клонирования:", result.stderr)
    raise RuntimeError("Не удалось клонировать репозиторий.")
print("✅ Репозиторий клонирован")

## 🔑 Секреты и проверка Яндекс.Диска

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"]     = userdata.get("HF_TOKEN")
os.environ["YANDEX_TOKEN"] = userdata.get("YANDEX_TOKEN")

# ── Передаём настройки пайплайну ──
env = {
    "PIPELINE_MODE":             "yadisk",
    "PIPELINE_VIDEOS_FOLDER":    VIDEOS_FOLDER,
    "PIPELINE_OUTPUT_FOLDER":    OUTPUT_FOLDER,
    "PIPELINE_WORK_DIR":         "/content/results",
    "PIPELINE_DOWNLOAD_DIR":     "/content/videos_tmp",
    "PIPELINE_DIARIZE":          str(DIARIZE).lower(),
    "PIPELINE_MAX_VIDEOS":       str(MAX_VIDEOS),
    "PIPELINE_WHISPER_MODEL":    WHISPER_MODEL,
    "PIPELINE_LANGUAGE":         LANGUAGE,
    "PIPELINE_MIN_SPEAKERS":     str(MIN_SPEAKERS),
    "PIPELINE_MAX_SPEAKERS":     str(MAX_SPEAKERS),
    "PIPELINE_MIN_DURATION_SEC": str(MIN_DURATION_SEC),
}
if INITIAL_PROMPT.strip():
    env["PIPELINE_INITIAL_PROMPT"] = INITIAL_PROMPT.strip()
os.environ.update(env)

# ── Проверка токена и что нашли в папке ──
import yadisk as yadisk_lib
yd = yadisk_lib.YaDisk(token=os.environ["YANDEX_TOKEN"])
if not yd.check_token():
    raise RuntimeError("YANDEX_TOKEN недействителен — проверь секрет в Colab.")
if not yd.exists(VIDEOS_FOLDER):
    raise RuntimeError(
        f"Папка {VIDEOS_FOLDER} не найдена на Яндекс.Диске. "
        "Создай её (или поменяй VIDEOS_FOLDER в настройках выше) и закинь туда видео.")

exts = {".mp4", ".mov", ".avi", ".mkv", ".webm", ".m4v", ".zip"}
def _count(folder, depth=0):
    if depth > 5:
        return []
    found = []
    for item in yd.listdir(folder, fields=["name", "path", "type"], limit=500):
        if item.type == "file" and os.path.splitext(item.name)[-1].lower() in exts:
            found.append(item.name)
        elif item.type == "dir" and not item.name.startswith("_"):
            found += _count(item.path, depth + 1)
    return found

found = _count(VIDEOS_FOLDER)
print(f"✅ Яндекс.Диск подключён. Видео в очереди: {len(found)}")
for f in sorted(found)[:20]:
    print(f"   • {f}")
if len(found) > 20:
    print(f"   ... и ещё {len(found) - 20}")
print(f"\nРезультат каждого видео будет рядом с ним же на Яндекс.Диске.")

## 🚀 Запуск
Если сессия оборвётся — просто запусти всё заново: этапы кэшируются, уже загруженные
на Диск результаты пропускаются (по файлу `transcription_progress.json` в VIDEOS_FOLDER),
недоделанные продолжатся с места остановки.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "src.runner"],
    cwd="/content/colab-drive-transcriber"
)
if result.returncode != 0:
    print(f"❌ Пайплайн завершился с ошибкой (код {result.returncode})")
else:
    print("\n🎉 Готово! Транскрипты (.docx) — рядом с каждым видео на Яндекс.Диске.")

## 📄 Что получается на выходе

Рядом с каждым видео на Яндекс.Диске (в той же папке, где лежало само видео) —
**`<имя видео>.docx`**: красивый читаемый транскрипт (как у SpeechToText) —
«Спикер N:» жирным, под ним абзацы `ЧЧ:ММ:СС - текст`.

Прогресс обработки хранится в `VIDEOS_FOLDER/transcription_progress.json` — повторный
запуск блокнота не станет транскрибировать уже готовые видео.

**Частые вопросы:**
- *Диаризация упала с ошибкой доступа* → проверь, что принял условия моделей pyannote (ссылки в шапке) тем же аккаунтом, чей `HF_TOKEN`.
- *Надо быстрее и спикеры не нужны* → `DIARIZE = False`.
- *Обработать заново уже готовое видео* → удали строку с его путём из `transcription_progress.json` на Яндекс.Диске (или сам файл, чтобы сбросить прогресс целиком).
- *YANDEX_TOKEN недействителен* → токен из oauth.yandex.ru может протухнуть; получи новый по той же ссылке.